# 🎓 Feynman Tutor Agent

> **리처드 파인만의 교육 철학을 구현한 AI 튜터** — LangGraph 기반 교육 에이전트

---

## 📋 Step 1: 에이전트 설계

### 이름
**Feynman Tutor** — 파인만 기법 기반 AI 교육 에이전트

### 목적
학습자가 모르는 개념을 입력하면, 파인만 기법(Feynman Technique)을 사용해 누구나 이해할 수 있는 쉬운 언어와 비유로 설명하고, 이해도 퀴즈로 학습 완성도를 확인하며, 틀릴 경우 최대 2회 재설명하는 교육 에이전트.

### 핵심 기능 (최소 3가지)
1. **Feynman 설명 생성**: 복잡한 개념을 5살 아이에게 설명하듯, 비유·예시를 포함해 쉽게 풀어서 설명
2. **이해도 퀴즈**: 설명한 내용을 기반으로 4지선다 퀴즈를 자동 생성하고 정답을 판별
3. **적응형 재설명**: 틀린 경우 다른 각도/비유로 재설명 (최대 2회), 최종 실패 시 추가 학습 자료 안내

### 파인만 기법 (Feynman Technique)이란?
> 리처드 파인만(Richard Feynman, 노벨물리학상 수상자)이 사용한 학습법:
> 1. 배우려는 개념을 선택한다
> 2. **아이에게 설명하듯** 쉬운 언어로 설명한다
> 3. 막히는 부분을 파악한다 (지식의 공백)
> 4. 다시 단순하게 설명한다

---

### 그래프 구조: 노드 & 엣지 다이어그램

```
┌─────────────────────────────────────────────────────┐
│               Feynman Tutor Graph                   │
│                                                     │
│  START                                              │
│    │                                                │
│    ▼                                                │
│  ┌─────────────────────────────┐                   │
│  │        explain_node         │ ← Node 1           │
│  │  파인만 스타일 설명 생성      │                   │
│  │  (비유, 단순 언어, 예시 포함) │                   │
│  └─────────────────────────────┘                   │
│    │                                                │
│    ▼                                                │
│  ┌─────────────────────────────┐                   │
│  │         quiz_node           │ ← Node 2           │
│  │  4지선다 퀴즈 생성 + 평가    │ ◄─────────┐       │
│  └─────────────────────────────┘           │       │
│    │                                       │       │
│    ├── ✅ is_correct=True ────────────► END (성공)  │
│    │                                                │
│    ├── ❌ is_correct=False                          │
│    │      & attempts >= max_retry ─────► END (실패) │
│    │                                                │
│    └── ❌ is_correct=False                          │
│           & attempts < max_retry                   │
│             │                                      │
│             ▼                                      │
│  ┌─────────────────────────────┐                   │
│  │       re_explain_node       │ ← Node 3           │
│  │  다른 비유로 재설명            │                   │
│  │  attempts += 1              │ ──────────────────┘│
│  └─────────────────────────────┘                   │
└─────────────────────────────────────────────────────┘
```

**노드 요약:**
| 노드 | 역할 | 입력 | 출력 |
|------|------|------|------|
| `explain_node` | 파인만 설명 생성 | concept | explanation |
| `quiz_node` | 퀴즈 생성 & 평가 | explanation | is_correct |
| `re_explain_node` | 재설명 (다른 각도) | explanation, attempts | new explanation |

---

## 🔧 Step 2: 기초 구축

### 환경 설정

In [ ]:
# 의존성 설치 (uv로 관리)
# 터미널에서: uv sync
# 또는 이 셀에서 직접:
# !uv sync

import os
import json
from dotenv import load_dotenv

load_dotenv()

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY 환경변수를 설정해주세요! (.env 파일 또는 환경변수)')

print('✅ 환경 설정 완료')
print(f'  API Key: {os.getenv("OPENAI_API_KEY")[:8]}...')

### State 정의

LangGraph에서 노드 간에 공유되는 상태(State)를 `TypedDict`로 정의합니다.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage


class TutorState(TypedDict):
    messages: Annotated[list, add_messages]  # 대화 이력 (자동 누적)
    concept: str          # 학습할 개념
    explanation: str      # 현재 생성된 설명
    quiz_question: str    # 퀴즈 문제
    quiz_choices: list    # 4지선다 선택지 리스트
    correct_index: int    # 정답 인덱스 (0-based)
    user_answer_idx: int  # 사용자가 선택한 인덱스 (0-based)
    is_correct: bool      # 정답 여부
    attempts: int         # 재설명 횟수 (0에서 시작)
    max_retry: int        # 최대 재시도 횟수 (default: 2)
    session_status: str   # 'in_progress' | 'success' | 'failed'


print('✅ TutorState 정의 완료')
print('필드:', list(TutorState.__annotations__.keys()))

### LLM 설정

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)
print(f'✅ LLM 설정 완료: {llm.model_name}')

### Node 1: `explain_node` — 파인만 설명 생성

사용자가 입력한 개념을 파인만 기법으로 설명합니다.
- 5살 아이에게 설명하듯 쉬운 언어 사용
- 실생활 비유와 예시 2개 이상 포함
- 한 문장 핵심 요약으로 마무리

In [ ]:
def explain_node(state: TutorState) -> dict:
    """Node 1: 파인만 기법으로 개념 설명 생성"""
    concept = state['concept']

    system_prompt = '''당신은 리처드 파인만처럼 어떤 복잡한 개념도 누구나 이해할 수 있게 설명하는 AI 튜터입니다.

파인만 기법 원칙:
1. 5살 아이에게 설명하듯 쉬운 일상 언어를 사용하세요
2. 전문 용어는 반드시 쉽게 풀어서 설명하세요
3. 실생활의 구체적인 비유와 예시를 2개 이상 포함하세요
4. 핵심 개념을 단계별로 논리적으로 설명하세요
5. 마지막에 "💡 한 문장 요약:"으로 마무리하세요'''

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f'다음 개념을 파인만 기법으로 설명해주세요: {concept}')
    ])

    explanation = response.content

    print(f'\n{"="*60}')
    print(f'🎓 Feynman Tutor: [{concept}] 설명')
    print(f'{"="*60}')
    print(explanation)

    return {
        'explanation': explanation,
        'session_status': 'in_progress',
        'messages': [AIMessage(content=explanation)],
    }


print('✅ explain_node 정의 완료')

### Node 2: `quiz_node` — 이해도 퀴즈 생성 & 평가

설명을 기반으로 4지선다 퀴즈를 자동 생성하고, 사용자 답변을 평가합니다.

In [ ]:
def quiz_node(state: TutorState) -> dict:
    """Node 2: 4지선다 퀴즈 생성 및 사용자 답변 평가"""
    concept = state['concept']
    explanation = state['explanation']
    attempts = state.get('attempts', 0)

    # 퀴즈 생성 프롬프트
    quiz_prompt = f'''다음 설명을 바탕으로 이해도를 확인하는 4지선다 퀴즈를 1개 만들어주세요.

개념: {concept}
설명: {explanation}

반드시 아래 JSON 형식으로만 응답하세요 (다른 텍스트 없이):
{{
  "question": "퀴즈 문제 (한국어)",
  "choices": ["선택지1", "선택지2", "선택지3", "선택지4"],
  "correct_index": 0,
  "answer_explanation": "정답 해설"
}}

correct_index는 0~3 사이의 정수입니다. 핵심 개념 이해도를 측정하는 문제로 만들어주세요.'''

    response = llm.invoke([
        SystemMessage(content='교육 평가 전문가입니다. 요청한 JSON 형식으로만 응답합니다.'),
        HumanMessage(content=quiz_prompt)
    ])

    # JSON 파싱
    try:
        content = response.content.strip()
        if '```json' in content:
            content = content.split('```json')[1].split('```')[0].strip()
        elif '```' in content:
            content = content.split('```')[1].split('```')[0].strip()
        quiz_data = json.loads(content)
    except Exception:
        # 파싱 실패 시 기본 퀴즈 사용
        quiz_data = {
            'question': f'{concept}의 핵심 특징으로 올바른 것은?',
            'choices': ['위 설명에서 설명한 핵심 내용', '관련 없는 설명 A', '관련 없는 설명 B', '관련 없는 설명 C'],
            'correct_index': 0,
            'answer_explanation': '설명에서 제시한 핵심 내용이 정답입니다.',
        }

    # 퀴즈 출력
    trial_num = attempts + 1
    max_trials = state['max_retry'] + 1
    print(f'\n{"="*60}')
    print(f'📝 이해도 확인 퀴즈 ({trial_num}/{max_trials}번째 시도)')
    print(f'{"="*60}')
    print(f'\n{quiz_data["question"]}\n')
    for i, choice in enumerate(quiz_data['choices']):
        print(f'  {i + 1}. {choice}')

    # 사용자 입력
    while True:
        try:
            user_input = input('\n답을 입력하세요 (1-4): ').strip()
            user_idx = int(user_input) - 1
            if 0 <= user_idx <= 3:
                break
            print('  ⚠️  1에서 4 사이의 숫자를 입력해주세요.')
        except ValueError:
            print('  ⚠️  숫자를 입력해주세요.')

    # 정답 확인
    correct_idx = quiz_data['correct_index']
    is_correct = (user_idx == correct_idx)

    if is_correct:
        print(f'\n✅ 정답입니다!')
        print(f'   {quiz_data["answer_explanation"]}')
    else:
        print(f'\n❌ 틀렸습니다. 정답은 {correct_idx + 1}번이었습니다.')
        print(f'   {quiz_data["answer_explanation"]}')

    return {
        'quiz_question': quiz_data['question'],
        'quiz_choices': quiz_data['choices'],
        'correct_index': correct_idx,
        'user_answer_idx': user_idx,
        'is_correct': is_correct,
        'messages': [AIMessage(content=f'퀴즈: {quiz_data["question"]}')],
    }


print('✅ quiz_node 정의 완료')

### Node 3: `re_explain_node` — 적응형 재설명

퀴즈에서 틀렸을 때, 완전히 다른 비유와 다른 각도로 재설명합니다.

In [ ]:
def re_explain_node(state: TutorState) -> dict:
    """Node 3: 다른 비유/각도로 재설명"""
    concept = state['concept']
    prev_explanation = state['explanation']
    attempts = state.get('attempts', 0)

    print(f'\n{"="*60}')
    print(f'🔄 다른 방식으로 다시 설명해볼게요! (재시도 {attempts + 1}/{state["max_retry"]})')
    print(f'{"="*60}')

    system_prompt = '''당신은 파인만 기법으로 가르치는 AI 튜터입니다.
이전 설명으로 학습자가 이해하지 못했습니다.
반드시 이전과 완전히 다른 방식으로 재설명하세요.

재설명 원칙:
- 완전히 다른 실생활 비유를 사용하세요 (이전 비유 재사용 금지)
- 더 단순하고 짧은 문장을 사용하세요
- 다른 관점(왜? 어떻게? 무엇이?)에서 접근하세요
- 시각적 이미지를 떠올릴 수 있는 묘사를 포함하세요
- 마지막에 "💡 한 문장 요약:"으로 마무리하세요'''

    prompt = f'''개념: {concept}

이전 설명 (이 방식과 다르게 설명하세요):
{prev_explanation}

위와 완전히 다른 비유와 방식으로 더 쉽게 설명해주세요.'''

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt)
    ])

    new_explanation = response.content
    print(new_explanation)

    return {
        'explanation': new_explanation,
        'attempts': attempts + 1,
        'messages': [AIMessage(content=new_explanation)],
    }


print('✅ re_explain_node 정의 완료')

### 조건부 엣지 — 라우팅 함수

퀴즈 결과에 따라 다음 노드를 결정하는 라우팅 함수입니다.

In [ ]:
def route_after_quiz(state: TutorState) -> Literal['re_explain_node', '__end__']:
    """퀴즈 결과에 따른 라우팅: 정답→END, 오답+여유→재설명, 오답+초과→END"""
    if state['is_correct']:
        print(f'\n🎉 학습 완료! [{state["concept"]}] 개념을 성공적으로 이해했습니다!')
        print('   Feynman Tutor와 함께한 학습을 마칩니다. 수고하셨습니다! 👏')
        return '__end__'

    attempts = state.get('attempts', 0)
    if attempts >= state['max_retry']:
        print(f'\n💪 [{state["concept"]}] 개념이 아직 어렵죠? 괜찮아요!')
        print('\n📖 추가 학습 자료를 참고해보세요:')
        concept = state['concept']
        print(f'  - Wikipedia (한국어): https://ko.wikipedia.org/wiki/{concept}')
        print(f'  - YouTube 검색: "{concept} 쉽게 설명"')
        print(f'  - Khan Academy: https://ko.khanacademy.org')
        print('\n  천천히 여러 번 접하다 보면 반드시 이해됩니다! 🌱')
        return '__end__'

    return 're_explain_node'


print('✅ route_after_quiz 정의 완료')

### 그래프 조립 — LangGraph StateGraph

3개의 노드와 조건부 엣지를 LangGraph로 연결합니다.

In [ ]:
from langgraph.graph import StateGraph, END


def build_tutor_graph():
    graph = StateGraph(TutorState)

    # 노드 등록
    graph.add_node('explain_node', explain_node)
    graph.add_node('quiz_node', quiz_node)
    graph.add_node('re_explain_node', re_explain_node)

    # 진입점 설정
    graph.set_entry_point('explain_node')

    # 기본 엣지: explain → quiz
    graph.add_edge('explain_node', 'quiz_node')

    # 조건부 엣지: quiz → (정답:END | 재시도초과:END | 오답:re_explain)
    graph.add_conditional_edges(
        'quiz_node',
        route_after_quiz,
        {
            're_explain_node': 're_explain_node',
            '__end__': END,
        }
    )

    # 재설명 후 다시 퀴즈로
    graph.add_edge('re_explain_node', 'quiz_node')

    return graph.compile()


tutor_graph = build_tutor_graph()
print('✅ Feynman Tutor Graph 컴파일 완료!')
print(f'   노드: explain_node → quiz_node ⇄ re_explain_node')
print(f'   조건부 엣지: quiz_node → [성공:END | 재시도:re_explain_node | 실패:END]')

### 그래프 시각화

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(tutor_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print('그래프 PNG 렌더링 불가 (graphviz 미설치 시 정상)')
    print('\n📊 Mermaid 다이어그램:')
    print(tutor_graph.get_graph().draw_mermaid())

---

## 🚀 데모 실행

아래 `concept` 변수를 원하는 개념으로 바꾸고 실행하세요!

**예시:** `'블록체인'`, `'양자 얽힘'`, `'재귀 함수'`, `'미적분'`, `'자연 선택'`

In [ ]:
def run_feynman_tutor(concept: str, max_retry: int = 2) -> dict:
    """Feynman Tutor 에이전트 실행

    Args:
        concept: 학습할 개념 (예: '블록체인', '재귀 함수')
        max_retry: 최대 재설명 횟수 (default: 2)

    Returns:
        최종 TutorState
    """
    print(f'\n{"🎓" * 20}')
    print('Feynman Tutor Agent 시작!')
    print(f'학습할 개념: {concept}')
    print(f'최대 재시도 횟수: {max_retry}')
    print(f'{"🎓" * 20}')

    initial_state: TutorState = {
        'messages': [],
        'concept': concept,
        'explanation': '',
        'quiz_question': '',
        'quiz_choices': [],
        'correct_index': 0,
        'user_answer_idx': -1,
        'is_correct': False,
        'attempts': 0,
        'max_retry': max_retry,
        'session_status': 'in_progress',
    }

    final_state = tutor_graph.invoke(initial_state)
    return final_state


print('✅ run_feynman_tutor() 함수 준비 완료')
print('\n아래 셀을 실행해서 데모를 시작하세요!')

In [ ]:
# ✏️ 배우고 싶은 개념을 입력하세요!
concept = '블록체인'  # ← 여기를 바꿔보세요

result = run_feynman_tutor(concept, max_retry=2)